# 噴塗產線系統：Service 與 API 架構整理

本 Notebook 整理目前正式主線會使用的 Service 與 API，暫時不納入仍在更新中的 Manager UI 專用介面。

核心原則：

```text
Service：負責資料查詢、製程指標計算、狀態判斷、告警與預測
API：負責接收參數、呼叫 Service、整理 JSON 給 UI
UI：負責畫面顯示、操作與圖表呈現
```


## 1. 整體架構

```text
RawData / DataProcess
        ↓
PostgreSQL Database
        ↓
Integrated / Monitoring / Future / Troubleshooting Service
        ↓
FastAPI
        ↓
Engineer UI / Manager UI
```

Ontology 規則會在 Integrated Service 與 Monitoring Service 內部被使用，用來判斷：

```text
normal
warning
fault
```


## 2. 核心 Service

| Service | 主要輸入 | 主要工作 | 主要輸出／回寫 |
|---|---|---|---|
| **Integrated Service** | `sensor_1min`、`sensor_3min`、`batch_run`、TimeMode／BatchMode 參數 | 查詢過去、現在或批次資料；計算流量、空壓、噴幅、濾網堵塞、估測膜厚、Qc、Quality Score；組合元件與站點狀態 | `current_snapshot`、`metrics`、`components`、`time_series`、`viewer_state`、`batch_axis`；指定 `write_back=true` 時可回寫 `batch_station_status` |
| **Event Rule／Runtime Rule Classifier** | 感測欄位與量測值 | 優先讀取 Ontology TTL，判斷 `normal / warning / fault`；Ontology 失敗時使用 JSON threshold fallback | 狀態、規則來源、原因 ID、處置 ID；供 Integrated 與 Monitoring 使用 |
| **Monitoring Service／Worker** | 新增的感測資料、資料品質標記、門檻判斷結果 | 掃描資料、判斷異常、避免重複告警；`interpolated` 資料可跳過或降低告警 | 寫入 `alert_event`；更新 `batch_station_status` |
| **Future Service** | anchor batch／目前站點結果、未來時間、品質分數 | 建立簡易 deterministic projection；計算預測良率、預測 NG、風險等級 | Future payload；需要保存時寫入 `future_prediction_result` |
| **Troubleshooting Service** | 元件、狀態、原因、站點 | 查詢可能原因、處理建議、預估停機時間與所需技能 | 故障排查矩陣與 recommendation |
| **Database VersionB Adapter** | Service 的 DB 查詢／寫入要求 | 統一連接 PostgreSQL，包裝原始資料查詢、告警、狀態與預測資料存取 | 提供上述 Service 使用；本身不負責製程演算法 |


## 3. Service 分工重點

```text
Integrated Service
→ 計算畫面需要的數值與站點狀態

Monitoring Service
→ 產生真正的告警並寫入 Database

Future Service
→ 產生未來品質與風險預估

Troubleshooting Service
→ 提供異常原因與處置建議

API
→ 呼叫 Service、整理欄位與回傳 JSON，不重新計算製程公式
```


## 4. UI 查詢類 API

這一組主要提供 UI 顯示使用，預設為 **read-only，不回寫 Database**。

| API | 使用的 Service | 功能 | 是否回寫 DB |
|---|---|---|---|
| `POST /api/time-series` | Integrated Service | 取得完整核心 Service 結果，包含時間序列、metrics、components 等 | 否 |
| `POST /api/time-series/ui/summary` | Integrated Service | 取得三站摘要、正常／警告／異常數量、平均品質與風險摘要 | 否 |
| `POST /api/time-series/ui/station-detail` | Integrated Service | 取得單站目前值、品質指標、元件狀態、製程參數與時間資訊 | 否 |
| `POST /api/time-series/ui/component-detail` | Integrated Service＋Rule／Troubleshooting 資訊 | 取得單一元件數值、狀態、判斷依據、原因與建議 | 否 |
| `GET /api/database/status` | Database status module | 檢查 DB 連線、資料表是否存在、筆數與最新時間 | 否 |
| `GET /api/health` | API 本身 | 檢查 FastAPI 是否正常啟動 | 否 |


## 5. 告警類 API

告警 API 不會自己執行 Monitoring Service，而是讀取或修改 Database 中已存在的 `alert_event`。

| API | 資料來源 | 功能 | 是否回寫 DB |
|---|---|---|---|
| `GET /api/alerts` | `alert_event` | 查詢告警列表，可依站點、狀態、確認狀態與天數篩選 | 否 |
| `GET /api/alerts/unacknowledged/{station_id}` | `alert_event` | 查詢指定站點未確認告警 | 否 |
| `GET /api/alerts/{event_id}` | `alert_event`＋原因／對策關聯 | 取得單一告警詳細資料 | 否 |
| `GET /api/alerts/causes/{cause_id}/responses` | 原因與處置 catalog | 依原因取得建議對策 | 否 |
| `PATCH /api/alerts/{event_id}/acknowledge` | `alert_event` | 將告警標記為已確認 | **是，只更新確認時間** |

完整告警流程：

```text
sensor_1min / sensor_3min
        ↓
Monitoring Service
        ↓
Ontology / Threshold 判斷
        ↓
寫入 alert_event
        ↓
GET /api/alerts
        ↓
UI 顯示
```


## 6. Service Orchestration API

這一組用來直接執行 Service、測試或明確回寫 Database，不是一般 UI 每次重新整理都要呼叫。

| API | 使用的 Service | 功能 | 是否回寫 DB |
|---|---|---|---|
| `GET /api/service-orchestration/status` | 所有 Service 模組 | 檢查 Integrated、Future、Monitoring、Troubleshooting 與 DB Adapter 是否可用 | 否 |
| `POST /api/service-orchestration/integrated/query` | Integrated Service | 執行一次整合查詢；可透過 `write_back` 決定是否保存 | 預設否，可選擇 |
| `POST /api/service-orchestration/integrated/run-once` | Integrated Service | 強制以 `write_back=true` 執行整合計算 | **是** |
| `POST /api/service-orchestration/monitoring/run` | Monitoring Service | 掃描感測資料，產生告警與更新站點狀態 | **是** |
| `POST /api/service-orchestration/future/payload` | Future Service | 只建立 future prediction payload，不保存 | 否 |
| `POST /api/service-orchestration/future/save` | Future Service | 建立並保存 Future prediction | **是，寫入 `future_prediction_result`** |
| `GET /api/service-orchestration/troubleshooting/matrix` | Troubleshooting Service | 查詢故障排查矩陣 | 否 |
| `GET /api/service-orchestration/troubleshooting/states/{state}/recommendations` | Troubleshooting Service | 依狀態查詢處理建議 | 否 |


## 7. Engineer UI 主要使用 API

| Engineer UI 功能 | 主要 API |
|---|---|
| 首頁三站總覽 | `POST /api/time-series/ui/summary` |
| 每一站卡片與數值 | `POST /api/time-series/ui/station-detail` |
| 元件卡片與詳細判斷 | `POST /api/time-series/ui/component-detail` |
| 趨勢圖 | `station-detail` 的 `time_series`，不足時再以 `component-detail` 多時間點查詢 |
| DB 連線顯示 | `GET /api/database/status` |
| API 狀態 | `GET /api/health` |


## 8. 目前先不列入正式主線

| 項目 | 原因 |
|---|---|
| `GET /api/v1/bundle` | 不是目前確認要採用的主要 API，可先視為舊整合或保留 route |
| Manager 專用 `/api/manager/dashboard` | 是舊 Manager UI 自己預留的 API，不是正式主線 |
| `/api/v1/lines/{line_id}/...` 系列 | 屬於舊 Manager UI 預留 schema，等新版 Manager UI 確定後再決定 |
| Demo endpoint／demo fallback | 正式版本已移除，不應出現在正式架構表 |
| mock data API | 僅供舊 UI 展示，不列入正式資料流 |


## 9. 最終精簡表

| 類型 | 正式內容 |
|---|---|
| **Service** | Integrated、Event Rule／Ontology、Monitoring、Future、Troubleshooting |
| **UI API** | time-series、summary、station-detail、component-detail |
| **Alert API** | alerts、alert detail、acknowledge、cause responses |
| **執行／回寫 API** | integrated run-once、monitoring run、future save |
| **狀態 API** | health、database status、service orchestration status |

目前可以先以這一張表作為正式主線。等 Manager UI 新版本確定後，再另外補上「Manager UI 使用情況」，不需要先修改現有 Service 與 API 架構。


In [ ]:
# 快速參考：目前正式主線 API
official_apis = {
    "UI查詢": [
        "POST /api/time-series",
        "POST /api/time-series/ui/summary",
        "POST /api/time-series/ui/station-detail",
        "POST /api/time-series/ui/component-detail",
    ],
    "告警": [
        "GET /api/alerts",
        "GET /api/alerts/unacknowledged/{station_id}",
        "GET /api/alerts/{event_id}",
        "GET /api/alerts/causes/{cause_id}/responses",
        "PATCH /api/alerts/{event_id}/acknowledge",
    ],
    "Service Orchestration": [
        "GET /api/service-orchestration/status",
        "POST /api/service-orchestration/integrated/query",
        "POST /api/service-orchestration/integrated/run-once",
        "POST /api/service-orchestration/monitoring/run",
        "POST /api/service-orchestration/future/payload",
        "POST /api/service-orchestration/future/save",
        "GET /api/service-orchestration/troubleshooting/matrix",
        "GET /api/service-orchestration/troubleshooting/states/{state}/recommendations",
    ],
    "系統狀態": [
        "GET /api/health",
        "GET /api/database/status",
    ],
}

for category, apis in official_apis.items():
    print(f"\n[{category}]")
    for api in apis:
        print("-", api)
